In [9]:
from hydra import compose, initialize
from omegaconf import OmegaConf
import json
import os
from pathlib import Path
import glob
with initialize(config_path="../../../configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg.paths))

checkpoints: ${paths.results}/checkpoints
totalsegmri: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalsegmri
totalseg: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalseg
nako: /nfs/data/nii/data0/GNC/GNC_759
nnunet: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/results/nnunet
results: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/results/patch_icl



/tmp/ipykernel_6869/2997671192.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="../../../configs"):


In [11]:
# get predifined test split
splits_path = '/nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/splits_966.json'
with open(splits_path, 'r') as f:
    splits = json.load(f)
subjects_test = splits['test']

In [12]:
subjects_test

['100161',
 '100963',
 '100263',
 '100847',
 '100784',
 '100193',
 '100383',
 '100235',
 '100500',
 '100256',
 '100651',
 '100824',
 '100039',
 '100374',
 '100082',
 '100668',
 '100312',
 '100229',
 '100280',
 '100785',
 '100020',
 '100334',
 '100630',
 '100697',
 '100952',
 '100703',
 '100042',
 '100882',
 '100822',
 '100831',
 '100024',
 '100996',
 '100411',
 '100911',
 '100144',
 '100352',
 '100449',
 '100172',
 '100836',
 '100682',
 '100204',
 '100481',
 '100584',
 '100313',
 '100892',
 '100281',
 '100324',
 '100295',
 '100821',
 '100163',
 '100978',
 '100834',
 '100243',
 '100993',
 '100770',
 '100278',
 '100679',
 '100922',
 '100098',
 '100721',
 '100589',
 '100744',
 '100739',
 '100745',
 '100855',
 '100896',
 '100341',
 '100986',
 '100234',
 '100688',
 '100289',
 '100666',
 '100796',
 '100009',
 '100227',
 '100228',
 '100065',
 '100041',
 '100473',
 '100121',
 '100302',
 '100272',
 '100032',
 '100444',
 '100981',
 '100752',
 '100407',
 '100660',
 '100451',
 '100691',
 '100176',

In [ ]:
import nibabel as nib
import numpy as np
import shutil

def process_subject(subject, img_base_path, mask_base_path, img_target, mask_target, images_dir, labels_dir):
    img_path = img_base_path / subject / img_target
    mask_path = mask_base_path / subject / mask_target

    # --- Images: split 4D -> 4x 3D channel files ---
    print(f"Looking for images in: {img_path}")
    files = glob.glob(str(img_path))
    print(f"Subject: {subject}, Files found: {len(files)}")

    if len(files) > 0:
        img = nib.load(files[0])
        data = img.get_fdata()  # (320, 260, 316, 4)

        if data.ndim == 4:
            for ch in range(data.shape[-1]):
                channel_data = data[..., ch].astype(np.float32)
                new_img = nib.Nifti1Image(channel_data, img.affine, img.header)
                new_img.header.set_data_shape(channel_data.shape)
                dst = os.path.join(images_dir, f"{subject}_{ch:04d}.nii.gz")
                nib.save(new_img, dst)
                print(f"  Saved channel {ch} -> {dst}")
        else:
            # skip if not 4D
            print(f"  Warning: Image is not 4D, skipping: {files[0]}")

    # --- Mask: enforce 3D header and save ---
    print(f"Looking for masks in: {mask_path}")
    mask_files = glob.glob(str(mask_path))
    print(f"Subject: {subject}, Mask files found: {len(mask_files)}")

    if len(mask_files) > 0:
        mask = nib.load(mask_files[0])
        mask_data = mask.get_fdata().astype(np.uint8)

        # Squeeze out any phantom 4th dim
        if mask_data.ndim == 4:
            mask_data = mask_data[..., 0]

        new_mask = nib.Nifti1Image(mask_data, mask.affine, mask.header)
        new_mask.header.set_data_shape(mask_data.shape)
        dst = os.path.join(labels_dir, f"{subject}.nii.gz")
        nib.save(new_mask, dst)
        print(f"  Saved mask -> {dst}")


from concurrent.futures import ProcessPoolExecutor, as_completed

def run_parallel(subjects, images_dir, labels_dir, max_workers=16):
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                process_subject,
                subject,
                img_base_path, mask_base_path,
                img_target, mask_target,
                images_dir, labels_dir,
            ): subject
            for subject in subjects
        }
        for future in as_completed(futures):
            subject = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"ERROR processing {subject}: {e}")

run_parallel(subjects_test,
             os.path.join(raw_ds_dir, "imagesTs"),
             os.path.join(raw_ds_dir, "labelsTs"))